# 03 — Descriptive Analysis
Quintile bins, Table A & B, delta_shot_real, delta_goal_real, Fig 2 & 3.

In [30]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import os

from config import INTERIM_DIR, TABLES_DIR
from utils import make_quintile_bins, compute_delta, summarize_momentum_table
from plots import plot_momentum_bins

os.makedirs(TABLES_DIR, exist_ok=True)

In [31]:
# Load interim data
team_windows_df = pd.read_csv(os.path.join(INTERIM_DIR, 'team_windows.csv'))
print(f'Loaded team_windows_df: {len(team_windows_df)} rows')

Loaded team_windows_df: 10368 rows


In [32]:
# Quintile bins
team_windows_df['quintile_bin'] = make_quintile_bins(team_windows_df['rolling_xg_5'])
print('Quintile bin distribution:')
print(team_windows_df['quintile_bin'].value_counts().sort_index())

Quintile bin distribution:
quintile_bin
0    8295
1    2073
Name: count, dtype: int64


In [33]:
# Table A: momentum summary
table_a = summarize_momentum_table(team_windows_df)
print('=== Table A: Momentum Summary ===')
print(table_a.to_string(index=False))

table_a.to_csv(os.path.join(TABLES_DIR, 'table_a_momentum.csv'), index=False)
print('\nSaved to outputs/tables/table_a_momentum.csv')

=== Table A: Momentum Summary ===
 quintile_bin  n_rows  rolling_xg_5_mean  P_shot_next_2  P_goal_next_5
            0    8295           0.008761       0.180229       0.057143
            1    2073           0.264963       0.209358       0.097443

Saved to outputs/tables/table_a_momentum.csv


In [34]:
# Table B: breakdown by game state
table_b_rows = []
for gs, grp in team_windows_df.groupby('game_state'):
    grp = grp.copy()
    grp['qbin'] = make_quintile_bins(grp['rolling_xg_5'])
    agg = grp.groupby('qbin').agg(
        n=('shot_next_2','count'),
        P_shot_next_2=('shot_next_2','mean'),
        P_goal_next_5=('goal_next_5','mean'),
    ).reset_index()
    agg['game_state'] = gs
    table_b_rows.append(agg)

table_b = pd.concat(table_b_rows, ignore_index=True)
print('=== Table B: Momentum by Game State ===')
print(table_b.to_string(index=False))
table_b.to_csv(os.path.join(TABLES_DIR, 'table_b_by_game_state.csv'), index=False)

=== Table B: Momentum by Game State ===
 qbin    n  P_shot_next_2  P_goal_next_5 game_state
    0 1463       0.176350       0.087491    leading
    1  486       0.193416       0.090535    leading
    2  484       0.185950       0.103306    leading
    0 4403       0.174427       0.046559       tied
    1 1099       0.232939       0.093722       tied
    0 1947       0.179250       0.052388   trailing
    1  486       0.234568       0.090535   trailing


In [35]:
# Compute deltas
delta_shot_real = compute_delta(team_windows_df, 'shot_next_2')
delta_goal_real = compute_delta(team_windows_df, 'goal_next_5')
print(f'delta_shot_real (P(Q5) - P(Q1)): {delta_shot_real:.4f}')
print(f'delta_goal_real (P(Q5) - P(Q1)): {delta_goal_real:.4f}')

delta_shot_real (P(Q5) - P(Q1)): 0.0291
delta_goal_real (P(Q5) - P(Q1)): 0.0403


In [36]:
# Fig 2 & 3: Momentum bin plots
plot_momentum_bins(team_windows_df)

Saved → /Users/aidenpark/Project/cs109-soccer-momentum/outputs/figures/fig2_momentum_bins.png


## Supplementary Pressure Features

Corner shots, set-piece shots, and big chances (xG ≥ 0.30) in the last 5 minutes.

In [37]:
# Confirm supplementary columns are present
supp_cols = ['corner_shots_5', 'set_piece_shots_5', 'big_chances_5',
             'recent_corner_pressure', 'recent_set_piece_pressure', 'recent_big_chance_pressure']
missing = [c for c in supp_cols if c not in team_windows_df.columns]
if missing:
    print(f'WARNING: supplementary columns not found: {missing}')
    print('Re-run notebook 02 to rebuild team_windows with the new features.')
else:
    print('Supplementary columns present:', supp_cols)
    print(team_windows_df[supp_cols].describe().round(4))

Supplementary columns present: ['corner_shots_5', 'set_piece_shots_5', 'big_chances_5', 'recent_corner_pressure', 'recent_set_piece_pressure', 'recent_big_chance_pressure']
       corner_shots_5  set_piece_shots_5  big_chances_5  \
count      10368.0000         10368.0000     10368.0000   
mean           0.0899             0.1969         0.0422   
std            0.3489             0.4986         0.2078   
min            0.0000             0.0000         0.0000   
25%            0.0000             0.0000         0.0000   
50%            0.0000             0.0000         0.0000   
75%            0.0000             0.0000         0.0000   
max            3.0000             4.0000         2.0000   

       recent_corner_pressure  recent_set_piece_pressure  \
count              10368.0000                 10368.0000   
mean                   0.0723                     0.1580   
std                    0.2591                     0.3647   
min                    0.0000                     0.000

In [38]:
# Table S1: by recent_corner_pressure
tbl_s1 = (
    team_windows_df
    .groupby('recent_corner_pressure')
    .agg(n_rows=('shot_next_2', 'count'),
         P_shot_next_2=('shot_next_2', 'mean'),
         P_goal_next_5=('goal_next_5', 'mean'))
    .reset_index()
    .rename(columns={'recent_corner_pressure': 'corner_pressure_flag'})
)
tbl_s1['corner_pressure_flag'] = tbl_s1['corner_pressure_flag'].map({0: 'No', 1: 'Yes'})
print('=== Table S1: by Recent Corner Pressure ===')
print(tbl_s1.to_string(index=False))
tbl_s1.to_csv(os.path.join(TABLES_DIR, 'table_s1_corner_pressure.csv'), index=False)

=== Table S1: by Recent Corner Pressure ===
corner_pressure_flag  n_rows  P_shot_next_2  P_goal_next_5
                  No    9618       0.183926       0.063527
                 Yes     750       0.213333       0.086667


In [39]:
# Table S2: by recent_set_piece_pressure
tbl_s2 = (
    team_windows_df
    .groupby('recent_set_piece_pressure')
    .agg(n_rows=('shot_next_2', 'count'),
         P_shot_next_2=('shot_next_2', 'mean'),
         P_goal_next_5=('goal_next_5', 'mean'))
    .reset_index()
    .rename(columns={'recent_set_piece_pressure': 'set_piece_flag'})
)
tbl_s2['set_piece_flag'] = tbl_s2['set_piece_flag'].map({0: 'No', 1: 'Yes'})
print('=== Table S2: by Recent Set-Piece Pressure ===')
print(tbl_s2.to_string(index=False))
tbl_s2.to_csv(os.path.join(TABLES_DIR, 'table_s2_set_piece_pressure.csv'), index=False)

=== Table S2: by Recent Set-Piece Pressure ===
set_piece_flag  n_rows  P_shot_next_2  P_goal_next_5
            No    8730       0.181558       0.062772
           Yes    1638       0.210012       0.078144


In [40]:
# Table S3: by recent_big_chance_pressure
tbl_s3 = (
    team_windows_df
    .groupby('recent_big_chance_pressure')
    .agg(n_rows=('shot_next_2', 'count'),
         P_shot_next_2=('shot_next_2', 'mean'),
         P_goal_next_5=('goal_next_5', 'mean'))
    .reset_index()
    .rename(columns={'recent_big_chance_pressure': 'big_chance_flag'})
)
tbl_s3['big_chance_flag'] = tbl_s3['big_chance_flag'].map({0: 'No', 1: 'Yes'})
print('=== Table S3: by Recent Big Chance Pressure ===')
print(tbl_s3.to_string(index=False))
tbl_s3.to_csv(os.path.join(TABLES_DIR, 'table_s3_big_chance_pressure.csv'), index=False)

=== Table S3: by Recent Big Chance Pressure ===
big_chance_flag  n_rows  P_shot_next_2  P_goal_next_5
             No    9944       0.186142       0.064763
            Yes     424       0.183962       0.075472


In [41]:
# Table S4: big_chances_5 count bins (0 / 1 / 2+)
tw = team_windows_df.copy()
tw['big_chance_bin'] = tw['big_chances_5'].clip(upper=2).map({0: '0', 1: '1', 2: '2+'})
tbl_s4 = (
    tw.groupby('big_chance_bin')
    .agg(n_rows=('goal_next_5', 'count'),
         P_goal_next_5=('goal_next_5', 'mean'))
    .reset_index()
)
print('=== Table S4: P(goal next 5 min) by Big Chance Count Bin ===')
print(tbl_s4.to_string(index=False))
tbl_s4.to_csv(os.path.join(TABLES_DIR, 'table_s4_big_chance_bins.csv'), index=False)

# Table S5: set_piece_shots_5 count bins (0 / 1 / 2+)
tw2 = team_windows_df.copy()
tw2['sp_bin'] = tw2['set_piece_shots_5'].clip(upper=2).map({0: '0', 1: '1', 2: '2+'})
tbl_s5 = (
    tw2.groupby('sp_bin')
    .agg(n_rows=('shot_next_2', 'count'),
         P_shot_next_2=('shot_next_2', 'mean'),
         P_goal_next_5=('goal_next_5', 'mean'))
    .reset_index()
)
print('\n=== Table S5: Outcomes by Set-Piece Shot Count Bin ===')
print(tbl_s5.to_string(index=False))
tbl_s5.to_csv(os.path.join(TABLES_DIR, 'table_s5_set_piece_bins.csv'), index=False)

=== Table S4: P(goal next 5 min) by Big Chance Count Bin ===
big_chance_bin  n_rows  P_goal_next_5
             0    9944       0.064763
             1     410       0.075610
            2+      14       0.071429

=== Table S5: Outcomes by Set-Piece Shot Count Bin ===
sp_bin  n_rows  P_shot_next_2  P_goal_next_5
     0    8730       0.181558       0.062772
     1    1293       0.202630       0.067285
    2+     345       0.237681       0.118841
